# Arkenstone ARK-017 V2 — mechanism dissection

Pinned, fail-closed T4 launcher. V2 fixes sparse-replay treatment fidelity, strengthens fork/optimizer smoke checks, and preserves partial receipts to Google Drive while the run is active.

Use **Runtime → Change runtime type → T4 GPU**, then run cells top-to-bottom. Do not change the pinned commit or thresholds after seeing results.


In [ ]:
import os, shutil, subprocess, sys, time, torch
from pathlib import Path

PINNED_RUNNER_COMMIT = '377c4743f8017e3455f576eafb75bb8ab9c50284'
REPO = '/content/An-Ra-the-new-AGI-ark017-v2'
RESULTS = '/content/arkenstone_ark017_v2_results'
RUNNER = 'experiments/ARK-017/run_ark017_v2.py'
USE_DRIVE_BACKUP = True
BACKUP = '/content/drive/MyDrive/Arkenstone/ARK-017-V2_' + PINNED_RUNNER_COMMIT[:8]

assert torch.cuda.is_available(), 'Select a T4 GPU runtime before running.'
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)

if os.path.exists(REPO): shutil.rmtree(REPO)
if os.path.exists(RESULTS): shutil.rmtree(RESULTS)
subprocess.run(['git','clone','--branch','Arkenstone','--depth','80','https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git',REPO], check=True)
subprocess.run(['git','-C',REPO,'checkout','--detach',PINNED_RUNNER_COMMIT], check=True)
head = subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'], text=True).strip()
assert head == PINNED_RUNNER_COMMIT, (head, PINNED_RUNNER_COMMIT)

compile_targets = [
    RUNNER,
    'experiments/ARK-017/run_ark017.py',
    'experiments/ARK-014/run_ark014.py',
    'experiments/ARK-011/run_ark011.py',
    'experiments/COLAB/discovery_v7_common.py',
]
for rel in compile_targets:
    subprocess.run([sys.executable,'-m','py_compile',os.path.join(REPO,rel)], check=True)
print('PY_COMPILE PASS')

if USE_DRIVE_BACKUP:
    from google.colab import drive
    drive.mount('/content/drive')
    Path(BACKUP).mkdir(parents=True, exist_ok=True)
    print('Durable backup:', BACKUP)

smoke = subprocess.run([sys.executable, os.path.join(REPO,RUNNER), '--smoke-test', '--expected-head', PINNED_RUNNER_COMMIT], cwd=REPO)
assert smoke.returncode == 0, f'GPU smoke failed with code {smoke.returncode}'
print('ARK-017 V2 GPU SMOKE TEST PASS — safe to run full campaign')


In [ ]:
# Full campaign. 240 minutes is a safety cap, not a minimum.
import os, subprocess, sys, time

sync_proc = None
if USE_DRIVE_BACKUP:
    sync_cmd = f'''while true; do
      mkdir -p "{BACKUP}";
      cp -f {RESULTS}/*.json "{BACKUP}/" 2>/dev/null || true;
      cp -f {RESULTS}/*.zip "{BACKUP}/" 2>/dev/null || true;
      sleep 60;
    done'''
    sync_proc = subprocess.Popen(['bash','-lc',sync_cmd])

started = time.time()
try:
    run = subprocess.run([sys.executable, os.path.join(REPO,RUNNER), '--budget-minutes','240', '--expected-head',PINNED_RUNNER_COMMIT], cwd=REPO)
    print('FULL RUN RETURN CODE:', run.returncode)
finally:
    if sync_proc is not None:
        sync_proc.terminate()
        try: sync_proc.wait(timeout=5)
        except Exception: sync_proc.kill()
    if USE_DRIVE_BACKUP:
        subprocess.run(['bash','-lc',f'mkdir -p "{BACKUP}"; cp -f {RESULTS}/*.json "{BACKUP}/" 2>/dev/null || true; cp -f {RESULTS}/*.zip "{BACKUP}/" 2>/dev/null || true'])
print('Wall minutes:', (time.time()-started)/60.0)
assert run.returncode == 0, 'Runner failed. Download/upload the partial ZIP and failure receipt; do not rerun blindly.'


In [ ]:
from pathlib import Path
from google.colab import files

root = Path(RESULTS)
print('RESULT FILES:')
for p in sorted(root.glob('*')):
    if p.is_file(): print(f'{p.name:42s} {p.stat().st_size/1024:9.1f} KB')
zip_path = root / 'ARKENSTONE_ARK017_V2_RESULTS.zip'
assert zip_path.exists(), 'Result ZIP missing; inspect RESULT FILES and Drive backup.'
files.download(str(zip_path))
print('Also preserved in Drive:' if USE_DRIVE_BACKUP else 'Drive backup disabled', BACKUP if USE_DRIVE_BACKUP else '')
